# 第II部：Euler方程式の古典的数値解法

**比較対象：LF、LW（Richtmyerの2段階法）**

線形移流・Burgersとの関係を整理し、3成分の保存則をLF・LWで計算します。

[I 移流・Burgers](01_advection_burgers.ipynb) · [II Eulerの古典的解法](02_euler_classical.ipynb) · [III 近似リーマンソルバー](03_euler_riemann_solvers.ipynb) · [IV 高次精度化](04_euler_high_accuracy.ipynb)

数値流束・空間再構築・時間積分は区別して比較します。LWは滑らかな解で2次精度です。
第III部の「実用的な流束」と、第IV部の「高次精度化した構成」は排他的な分類ではありません。
表示名は `LLF (Rusanov)`、内部識別子は `LLF` です。

## 実行方法と設定

リポジトリ一式を取得し、`notebooks/` と `src/` の配置を保ってください。
計算は `src/euler1d_core.py`、比較・描画は `src/euler1d_study.py` を使用します。
設定セルを確認し、カーネルを再起動して上から実行してください。
`run_convergence` は収束試験、`run_extras` は追加実験、`run_shu_osher` はShu–Osher問題、`run_movies` は動画です。
動画は標準で有効です。追加実験・Shu–Osherは初期設定で無効です。

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/euler1d_study.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("リポジトリ一式を取得し、notebooks/ と src/ の配置を保って実行してください。")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from euler1d_core import PROBLEMS, run_unit_tests
from euler1d_study import PARTS, RUN_CACHE, get_run, run_part, summarize, show_table, plot_profiles, plot_pressure_minimum, run_convergence, shu_reference, export_results, make_part_animation

### 時間発展動画の問題を選択する

設定セルの `CFG["movie_problem"]` に、次のいずれかの文字列を指定します。

| 設定する値 | 問題 | 主な観察点 |
|:--|:--|:--|
| `entropy` | 滑らかなエントロピー波の移流 | 波形の拡散・位相誤差 |
| `contact` | 移動する接触不連続面 | 接触面の数値拡散・振動 |
| `contact_static` | 静止する接触不連続面 | 静止した密度不連続の保持 |
| `sod` | Sod衝撃波管 | 衝撃波・接触不連続面・希薄波 |
| `double_rarefaction` | 二重希薄波 | 低密度・低圧力での頑健性、計算停止 |
| `lax` | Lax衝撃波管 | Sodとは異なる衝撃波条件 |
| `transonic` | 音速をまたぐ希薄波 | エントロピー補正の影響 |

`shu_osher` は数値参照解を必要とするため、この簡易動画機能の選択肢には含めません。Shu–Osherの計算・静止図は `run_shu_osher` で実行できます。

動画を無効にする場合は `CFG["run_movies"] = False` に設定してください。

In [ ]:
PART = "classical"
CFG = {
    "gamma": 1.4,
    "cfl": 0.4,
    "n_basic": 200,
    "n_convergence": (100, 200, 400, 800),
    "limiter": "minmod",
    "wave_speeds": "einfeldt",
    "roe_entropy": 0.1,
    "n_shu_osher": 400,
    "n_reference": (1600, 3200),
    "reference_limiter": "mc",
    "run_convergence": True,
    "run_shu_osher": False,
    "run_extras": False,
    "run_movies": True,
    "movie_problem": "sod",  # 選択肢は直前の表を参照
    "movie_frames": 61,
    "export_results": True,
}
RUN_CACHE.clear()
RESULTS = {}
reference_fine = None
print("Python:", sys.version.split()[0], "NumPy:", np.__version__)
print(CFG)
OUTPUT_DIR = ROOT / "four_part_outputs" / PART
print("Part:", PARTS[PART]["title"])
print("Output directory:", OUTPUT_DIR)

## 1. 移流・BurgersからEuler方程式へ

### 1.1 移流、圧縮・膨張、圧力による加速

滑らかな1次元の理想気体では、物質微分$D/Dt=\partial_t+u\partial_x$を使って

$$\frac{D\rho}{Dt}=-\rho u_x,\qquad\frac{Du}{Dt}=-\frac{p_x}{\rho},\qquad\frac{Dp}{Dt}=-\gamma p u_x$$

と書けます。密度には圧縮・膨張、速度には圧力勾配、圧力には断熱的な圧縮・膨張が作用します。
これらは滑らかな領域の基本変数表示です。**衝撃波の計算には下に示す保存形を使います。**

$u=u_0,p=p_0$が一定なら、任意の密度分布$\rho_0(x-u_0t)$はEuler方程式の解になります。
$E=p_0/(\gamma-1)+\rho u_0^2/2$も併せて与えます。エントロピー波と接触面の移流は、流体方程式の中に線形移流を見いだせる例です。[R1]

一方、$u_t+u u_x=0$というBurgers形は速度の非線形移流を分離したモデルです。
$p_x=0$を初期に置くだけでは一般に圧力勾配ゼロを保てません。形式的な無圧力極限の滑らかな速度方程式と、衝撃波を含む圧縮性気体全体を混同しないでください。

### 1.2 3つの特性速度はどこから来るか

$W=(\rho,u,p)^T$に対して

$$W_t+B(W)W_x=0,\qquad B=\begin{pmatrix}u&\rho&0\\0&u&1/\rho\\0&\gamma p&u\end{pmatrix}.$$

$$\det(B-\lambda I)=(u-\lambda)\left[(u-\lambda)^2-\frac{\gamma p}{\rho}\right]=0.$$

したがって$c=\sqrt{\gamma p/\rho}$として、固有値は$u-c,u,u+c$です。
$a$が一つのスカラー移流や、速度$q$が一つのBurgersと異なり、複数の波を区別する必要があります。[R1]

### 1.3 一様背景まわりの線形化は、3本の移流方程式になる

$(\rho_0,u_0,p_0)$を一定とし、微小擾乱について

$$w_\pm=\delta u\pm\frac{\delta p}{\rho_0c_0},\qquad w_0=\delta\rho-\frac{\delta p}{c_0^2}$$

とおくと、

$$\partial_tw_\pm+(u_0\pm c_0)\partial_xw_\pm=0,\qquad\partial_tw_0+u_0\partial_xw_0=0.$$

$w_\pm$は両方向の音響波、$w_0$は接触・エントロピー波です。$u_0=0$でも音波は伝わります。
この独立な3本への分離は**一様背景に対する線形化**の結果です。非線形Euler全体を固定変数変換で独立な3本へ分けられるという意味ではありません。

第III部のRoe法は、このような局所的な特性分解を数値流束に利用します。
LF・LWは明示的な特性分解を使わず、同じ保存形の流束を3成分のベクトルへ拡張できます。

In [ ]:
rho0, u0, p0 = 1.0, 0.2, 1.0
c0 = np.sqrt(CFG["gamma"] * p0 / rho0)
B = np.array([[u0, rho0, 0.0], [0.0, u0, 1.0 / rho0], [0.0, CFG["gamma"] * p0, u0]])
L = np.array([[0.0, 1.0, -1.0 / (rho0 * c0)], [1.0, 0.0, -1.0 / c0**2], [0.0, 1.0, 1.0 / (rho0 * c0)]])
expected = np.array([u0 - c0, u0, u0 + c0])
np.testing.assert_allclose(np.sort(np.linalg.eigvals(B)), expected)
np.testing.assert_allclose(L @ B, np.diag(expected) @ L, atol=2e-14)
print("Characteristic speeds:", expected)

### 共通の方程式と保存形

無粘性・断熱の理想気体を扱います。量はすべて無次元です。$E$は**単位体積当たりの全エネルギー**、$e$は**単位質量当たりの内部エネルギー**です。

$$
\frac{\partial\boldsymbol U}{\partial t}+\frac{\partial\boldsymbol F}{\partial x}=0,
\qquad
\boldsymbol U=\begin{pmatrix}\rho\\\rho u\\E\end{pmatrix},
\qquad
\boldsymbol F=\begin{pmatrix}\rho u\\\rho u^2+p\\u(E+p)\end{pmatrix}.
$$
$$
E=\frac{p}{\gamma-1}+\frac12\rho u^2,\qquad
c=\sqrt{\frac{\gamma p}{\rho}},\qquad e=\frac{p}{(\gamma-1)\rho}.
$$

特性速度は$u-c,\ u,\ u+c$です。中央の波が接触・エントロピー波を担います。[R1]

コードでは`U.shape == (N, 3)`とし、`U[:, 0]`が密度、`U[:, 1]`が運動量密度、`U[:, 2]`が全エネルギー密度です。
基本変数`W`は$(\rho,u,p)$の順です。境界流束を差し引く更新式を全方式で共有します。

$$
\overline{\boldsymbol U}^{\,n+1}_i=\overline{\boldsymbol U}^{\,n}_i-
\frac{\Delta t}{\Delta x}(\boldsymbol F^*_{i+1/2}-\boldsymbol F^*_{i-1/2}).
$$

ここで格子上の未知量は**点値ではなくセル平均**です。
一様格子上で区分一定の状態を用いる場合、以下のLF/LWの保存形実装は対応する古典的な差分更新になります。

### 共通のテスト問題と初期条件

すべて$\gamma=1.4$の設定です。$W=(\rho,u,p)$と表記します。
矩形波のセル内占有率と正弦波のセル平均を解析的に求め、格子点で初期条件を単純にサンプルする誤差を避けます。

| キー | 領域・時刻 | 初期条件の要点 | 境界 |
|:--|:--|:--|:--|
| `entropy` | $[0,1],\ t=1$ | $\rho=1+0.2\sin(2\pi x),\ u=p=1$ | 周期 |
| `contact` | $[0,1],\ t=0.25$ | $0.25\le x<0.45$で$\rho=2$、他は1。$u=p=1$ | 周期 |
| `contact_static` | $[0,1],\ t=0.25$ | 同じ矩形密度、$u=0,\ p=1$ | 周期 |
| `sod` | $[0,1],\ x_0=0.5,\ t=0.2$ | $W_L=(1,0,1),\ W_R=(0.125,0,0.1)$ | 流出 |
| `double_rarefaction` | $[-0.5,0.5],\ x_0=0,\ t=0.15$ | $W_L=(1,-2,0.4),\ W_R=(1,2,0.4)$ | 流出 |
| `shu_osher` | $[-5,5],\ x_0=-4,\ t=1.8$ | 左は衝撃波通過後、右は$\rho=1+0.2\sin(5x),\ u=0,\ p=1$ | 左固定流入・右流出 |
| `transonic` | $[0,1],\ x_0=0.3,\ t=0.2$ | $W_L=(1,0.75,1),\ W_R=(0.125,0,0.1)$ | 流出 |
| `lax` | $[-0.5,0.5],\ x_0=0,\ t=0.14$ | $W_L=(0.445,0.698,3.528),\ W_R=(0.5,0,0.571)$ | 流出 |

SodとShu–Osherの設定は文献・公式テスト例と対応します。[R4, R5]
Shu–Osherの左状態は$\rho_L=27/7,\ u_L=20\sqrt{1.4}/9,\ p_L=31/3$とし、よく用いられる小数値の丸めを避けています。
$\gamma$を変更した場合、この左状態が同じMach数の衝撃波に対応するとは限りません。

流出境界は端のセルをゴーストセルにコピーする簡単な条件で、完全な無反射境界ではありません。
指定した終了時刻では、Sodなどのリーマン問題の主要波面は境界に達しません。

Shu–Osher・Lax・音速をまたぐ希薄波は追加実験で扱います。Laxはここでは**問題の名称**で、LF/LWだけを使うという意味ではありません。

### 厳密解と比較の基準

エントロピー波と接触不連続面は、そのまま速度$u$で平行移動します。
Sod・二重希薄波・追加のリーマン問題は、星領域の圧力$p_*$を

$$
f_L(p_*)+f_R(p_*)+u_R-u_L=0
$$

から二分法で求めます。希薄波と衝撃波で分岐する$f_K$、自己相似変数$\xi=(x-x_0)/t$における解をコードに明示しています。[R1]
ここでの「厳密解」は厳密なリーマン解の構造に基づく数値評価で、非線形方程式の解法と数値積分の許容誤差は残ります。

**誤差は保存変数をセル平均した厳密解と比較します。** 波面の位置で積分区間を分割してからGauss積分するので、不連続を横切る単純な台形積分は使いません。
速度と圧力の比較基準は$W(\overline U_{\rm exact})$です。これは一般に$\overline{W(U_{\rm exact})}$とは異なります。

図には見やすさのため点値の厳密解を実線で示しますが、誤差表にはセル平均基準を使います。
Shu–Osherは厳密解を用いず、後で別に高解像度の**数値参照解**を計算します。

この実装は**物理的な真空を含むリーマン解には対応していません**。
真空条件$u_R-u_L\ge2(c_L+c_R)/(\gamma-1)$を検出すると明示的に停止します。
ここで使う二重希薄波問題は真空条件を満たさず、中央の密度・圧力は正です。

### 時間刻み・破綻・保存則

時間刻みは各計算で

$$
\Delta t=C_{\rm CFL}\frac{\Delta x}{\max_i(|u_i|+c_i)}
$$

から求めます。**共通なのはCFL数と選択規則であり、時々刻々の$\Delta t$が全方式で同じとは限りません。**
終端で極端に短い最後のステップだけを取ると、LFでは$\Delta x/\Delta t$が急増します。
これを避けるため、残り時間がCFL上限の2ステップ以内になったら残りを1または2等分します。この規則は全方式に共通です。

非周期境界では領域内の総量が変化しても保存則違反とは限りません。$Q=\sum_iU_i\Delta x$として

$$
R_Q=Q(t)-Q(0)-\int_0^t(F_{\rm left}^*-F_{\rm right}^*)\,dt'
$$

を測定します。SSP-RK2の境界流束も段階の重み$1/2,1/2$で積算します。
表の`balance_max`は質量・運動量・全エネルギーの$|R_Q|$の最大値で、無次元の絶対残差です。
保存残差が小さいことだけで解の精度がよいとは判断できません。

`failed`の場合、`t_reached`は**最後に受理した時刻**です。
`failure_trial_end`は失敗した試行ステップの終端時刻で、中間段階での失敗なら実際の検出段階は`failure_reason`に表示されます。
失敗した計算の最終時刻誤差は計算しません。

### 評価指標

密度の誤差は、領域長$L$で規格化した平均$L_1$誤差

$$
\epsilon_1(\rho)=\frac1L\sum_i\Delta x\,|\overline\rho_i-\overline\rho_{i,{\rm exact}}|
$$

です。滑らかな周期波については、基本Fourierモードの振幅比`amp_ratio`と位相誤差`phase_rad`も測定します。
接触面の`rho_overshoot`は$\max(\rho)-2$の正の部分、`rho_undershoot`は$1-\min(\rho)$の正の部分です。
`mixed_length`は$1.1<\rho<1.9$のセル幅の合計で、二つの遷移領域と振動の影響を含む補助指標です。単一の接触面の厳密な10–90%幅ではありません。

図は岡部・伊藤パレット、英語ラベル、線種とマーカーの併用に統一します。
RoeとHLLCがほぼ重なること自体も、このテストでは重要な結果です。

### 計算本体の自己検証
9グループの基本検証を実行します。これは基本的な回帰テストであり、任意の初期条件での正確さ・正値性を保証するものではありません。

In [ ]:
checks = run_unit_tests(CFG["gamma"])
for i, name in enumerate(checks, 1):
    print(f"PASS {i}: {name}")
print("Validation groups:", len(checks))

<a id="part-2"></a>
# 第II部：古典的手法 — LFとLW

**問い：数値拡散による「なまり」と、数値分散による「振動」はどのように違うか。**

古典的LFの数値流束は
$$
F^*_{i+1/2}=\frac{F_i+F_{i+1}}2-\frac{\Delta x}{2\Delta t}(U_{i+1}-U_i)
$$
です。係数に$\Delta t$が含まれる点を、後のLLF (Rusanov)との比較で確認します。

ここでのLWはRichtmyerの2段階法です。
$$
U^{n+1/2}_{i+1/2}=\frac{U_i^n+U_{i+1}^n}{2}-\frac{\Delta t}{2\Delta x}(F_{i+1}^n-F_i^n),\qquad
F^*_{i+1/2}=F(U^{n+1/2}_{i+1/2}).
$$
滑らかな解での精度と、不連続付近での数値振動を別々の問題で観察します。
「古典的」は学習上の位置づけであり、LWが1次精度という意味ではありません。

### II.1 滑らかなエントロピー波

1周後の波形を厳密解と比較します。振幅の減衰・位相のずれを見ます。LWの滑らかな解に対する精度は、この問題で評価します。

In [ ]:
RESULTS[("classical", "entropy")] = run_part("classical", "entropy", CFG)
runs = RESULTS[("classical", "entropy")]
plot_profiles(runs, fields=('rho',), logarithmic=False, title_suffix=PARTS["classical"]["title"])
summarize(runs)

### II.2 静止する接触面

LWがこの特別な静止状態を保てても、移動する不連続面で非振動になるとは限りません。次の問題と対にして読んでください。

In [ ]:
RESULTS[("classical", "contact_static")] = run_part("classical", "contact_static", CFG)
runs = RESULTS[("classical", "contact_static")]
plot_profiles(runs, fields=('rho',), logarithmic=False, title_suffix=PARTS["classical"]["title"])
summarize(runs)

### II.3 移動する接触面

密度の矩形分布について、LFの平滑化とLWのovershoot/undershootを区別します。正確な移流位置を保つことと、値の範囲を守ることも別の検査です。

In [ ]:
RESULTS[("classical", "contact")] = run_part("classical", "contact", CFG)
runs = RESULTS[("classical", "contact")]
plot_profiles(runs, fields=('rho',), logarithmic=False, title_suffix=PARTS["classical"]["title"])
summarize(runs)

### II.4 Sod衝撃波管

密度・速度・圧力を分けて描き、なまりと振動が現れる場所を記録します。

In [ ]:
RESULTS[("classical", "sod")] = run_part("classical", "sod", CFG)
runs = RESULTS[("classical", "sod")]
plot_profiles(runs, fields=('rho', 'u', 'p'), logarithmic=False, title_suffix=PARTS["classical"]["title"])
summarize(runs)

### II.5 二重希薄波

低圧力での頑健性を調べます。停止した計算は最終時刻の図から除外し、表と圧力履歴に残します。破綻は隠さず比較結果として扱います。

In [ ]:
RESULTS[("classical", "double_rarefaction")] = run_part("classical", "double_rarefaction", CFG)
runs = RESULTS[("classical", "double_rarefaction")]
plot_profiles(runs, fields=('rho', 'p'), logarithmic=True, title_suffix=PARTS["classical"]["title"])
summarize(runs)
plot_pressure_minimum(runs)

### II.6 滑らかな解の収束次数

LFとLWの形式的な次数は異なります。誤差の大きさだけでなく、格子を細かくしたときの変化を測ります。

In [ ]:
if CFG["run_convergence"]:
    convergence_I = run_convergence("classical", CFG)

### 第II部のまとめ
LFでは振幅低下・接触面の広がり、LWでは滑らかな解の精度と不連続付近の振動を確認しました。
この時点では「小さい誤差」「非振動」「計算が停止しない」を一つの評価にまとめないでください。

## 時間発展の動画

動画は標準で表示します。`CFG["run_movies"] = False` にすると省略できます。`movie_problem` の問題を本冊の手法だけで比較します。初期値はSodです。
`FrameRecorder` を使い、フレーム出力のために数値積分の時間刻みを変更しません。
記録の有無で最終状態・ステップ数・境界流束が一致することを検査します。
**失敗した方式は停止時刻以降に描画しません。** 表示はJavaScriptアニメーションで、ffmpegは不要です。
この簡易動画はShu–Osherには対応しません。

In [ ]:
if CFG["run_movies"]:
    fig, movie = make_part_animation(PART, CFG["movie_problem"], CFG, n_frames=CFG["movie_frames"])
    display(HTML(movie.to_jshtml()))
    plt.close(fig)

## 任意：Shu–Osher問題

`CFG["run_shu_osher"] = True` で有効です。定義した初期条件と2解像度の数値参照解を用います。
黒線は**数値参照解であり厳密解ではありません**。粗い参照解と細かい参照解の差も表示します。
参照解を含め、必要な計算は本冊だけで実行します。図の比較対象は本冊の手法に限定します。
高解像度参照解は計算量が多いため、基本演習の後に実行してください。

In [ ]:
if CFG["run_shu_osher"]:
    reference_fine = shu_reference(CFG)
    runs = run_part(PART, "shu_osher", CFG, n=CFG["n_shu_osher"])
    RESULTS[(PART, "shu_osher")] = runs
    plot_profiles(runs, reference=reference_fine, xlim=(-1.0, 2.3), title_suffix=PARTS[PART]["title"])
    summarize(runs, reference=reference_fine)

## 任意：追加実験

`CFG["run_extras"] = True` で有効です。
Lax衝撃波管を本冊の手法で比較します。
LFのCFL数依存性も調べます。LFとLLF (Rusanov)の直接比較は第III部の任意実験で扱います。
Laxは**問題名**で、使うスキームをLF/LWに限定する意味ではありません。

In [ ]:
if CFG["run_extras"]:
    runs = run_part(PART, "lax", CFG)
    RESULTS[(PART, "lax")] = runs
    plot_profiles(runs, title_suffix=PARTS[PART]["title"])
    summarize(runs)
    cfl_runs = [get_run("entropy", "LF", CFG, cfl=c) for c in (0.2, 0.4, 0.8)]
    summarize(cfl_runs, columns=("method", "cfl", "n", "L1_rho", "amp_ratio", "phase_rad"))

## 結果の保存

本冊の計算結果を `four_part_outputs/` の **本冊専用サブフォルダー**に保存し、他の冊の結果を上書きしません。
同じ冊を同じ設定で実行し直すと、その冊の保存ファイルを更新します。
CSVに内部識別子 `solver`、表示名 `method`、学習分類 `part` を分けて記録します。
NPZの状態配列の番号はCSVの行順に対応します。停止した計算の誤差は最終時刻の誤差として扱いません。
参照解や部をまたぐ補助比較を有効にした場合は、それらの計算も本冊の出力に含みます。

In [ ]:
if CFG["export_results"]:
    saved_rows = export_results(OUTPUT_DIR, CFG, reference=reference_fine)
    failed_rows = [r for r in saved_rows if r["status"] != "ok"]
    print("Distinct runs:", len(saved_rows), "Recorded failures/incomplete runs:", len(failed_rows))
    show_table(failed_rows, ("part", "problem", "method", "order", "t_reached", "failure_trial_end", "failure_p_min"))

## 次の学習

[第III部](03_euler_riemann_solvers.ipynb)では、Roe・LLF (Rusanov)・HLL・HLLCを同じ1次精度の条件で比較します。
その後に、空間再構築と時間積分を一つずつ変える学習へ進みます。PPM・WENO・SSP-RK3等の比較は、次の学習課題です。

## 参考文献・実装の確認先

以下は原論文または開発者による公式資料です。Notebookのコードはこれらの数式・考え方を参考に、本教材用に組み立てた実装であり、Clawpack等を呼び出してはいません。

**[R1]** Clawpack, *The Euler equations of gas dynamics*：波の構造と厳密リーマン解。  
https://www.clawpack.org/riemann_book/html/Euler.html

**[R2]** Clawpack, *Approximate solvers for the Euler equations of gas dynamics*：Roe平均、HLLEの波速推定、低密度問題。  
https://www.clawpack.org/riemann_book/html/Euler_approximate.html

**[R3]** Clawpack, *Riemann Solver Package — Euler Equations*：HLLCの接触波速度と中間状態。  
https://www.clawpack.org/pyclaw/rp.html#euler-equations

**[R4]** FLASH User's Guide, *Hydrodynamics Test Problems*, §35.1.1：Sod問題と波面の位置。  
https://flash.rochester.edu/site/flashcode/user_support/flash_ug_devel/node191.html

**[R5]** Clawpack Gallery, *Shu–Osher problem*：初期状態、領域、終了時刻。  
https://www.clawpack.org/gallery/pyclaw/gallery/shocksine.html

**[R6]** Toro, E. F., Spruce, M., & Speares, W. (1994), *Restoration of the contact surface in the HLL-Riemann solver*, Shock Waves, 4, 25–34.  
https://doi.org/10.1007/BF01414629

**[R7]** Harten, A. (1983), *High resolution schemes for hyperbolic conservation laws*, Journal of Computational Physics, 49, 357–393.  
https://doi.org/10.1016/0021-9991(83)90136-5

**[R8]** Shu, C.-W., & Osher, S. (1988), *Efficient implementation of essentially non-oscillatory shock-capturing schemes*, Journal of Computational Physics, 77, 439–471.  
https://doi.org/10.1016/0021-9991(88)90177-5

**[R9]** Shu, C.-W., & Osher, S. (1989), *Efficient implementation of essentially non-oscillatory shock-capturing schemes, II*, Journal of Computational Physics, 83, 32–78.  
https://doi.org/10.1016/0021-9991(89)90222-2

初期条件・波速推定・エントロピー補正・リミッター・CFL数まで含めて、比較の設定を報告することが重要です。
「Roe」や「HLLC」という名前だけでは、実装のすべては決まりません。